# Run QC on raw single cell profiles

## Import libraries

In [3]:
import json
import logging
import os
import pathlib
import sys

import cosmicqc
import natsort
import numpy as np
import pandas as pd
from cytodataframe import CytoDataFrame
from timelapse_utils.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)
root_dir, in_notebook = init_notebook()
if in_notebook:
    import tqdm.notebook as tqdm
else:
    import tqdm

## Set paths and variables

In [4]:
image_base_dir = bandicoot_check(
    bandicoot_mount_path=pathlib.Path(f"{os.path.expanduser('~')}/mnt/bandicoot/"),
    root_dir=root_dir,
)
image_base_dir = root_dir / "data"
image_base_dir = pathlib.Path(f"{image_base_dir}/processed_data/").resolve(strict=True)
combined_profiles_path = image_base_dir / "5.combined_profiles"
qc_path = image_base_dir / "6.qc_profiles"
qc_path.mkdir(parents=True, exist_ok=True)

In [5]:
for file_path in tqdm.tqdm(
    combined_profiles_path.glob("*.parquet"),
    desc="Processing combined profiles",
    unit="file",
    leave=True,
):
    file_path_stem = file_path.stem
    qc_output_path = qc_path / f"{file_path_stem}_qc.parquet"
    continue

Processing combined profiles: 0file [00:00, ?file/s]

In [6]:
file_path

PosixPath('/home/lippincm/4TB_A/pyroptosis_live-cell_timelapse/data/processed_data/5.combined_profiles/C2_3.parquet')

In [13]:
df = pd.read_parquet(file_path)

In [ ]:
df[df.columns[df.isna().sum() > 0].tolist()]
df[[x for x in df.columns if "metadata" in x.lower()]]

,Metadata_Well,Metadata_Time,Metadata_Well_FOV,Metadata_FOV,Metadata_Well_FOV_Time,Metadata_ImageNumber,Metadata_Cells_Number_Object_Number,Metadata_Cytoplasm_Parent_Cells,Metadata_Cytoplasm_Parent_Nuclei,Metadata_Nuclei_Number_Object_Number,...,Metadata_Nuclei_Location_MaxIntensity_Y_CL488,Metadata_Nuclei_Location_MaxIntensity_Y_CL640,Metadata_Nuclei_Location_MaxIntensity_Y_NucleoLive,Metadata_Nuclei_Location_MaxIntensity_Y_SYTOXGreen,Metadata_Nuclei_Location_MaxIntensity_Z_BF,Metadata_Nuclei_Location_MaxIntensity_Z_CL488,Metadata_Nuclei_Location_MaxIntensity_Z_CL640,Metadata_Nuclei_Location_MaxIntensity_Z_NucleoLive,Metadata_Nuclei_Location_MaxIntensity_Z_SYTOXGreen,Metadata_single_cell_count
0,C2,1,C2_3,3,C2_3_1,1,1.0,1,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1133
1,C2,1,C2_3,3,C2_3_1,1,2.0,2,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1133
2,C2,1,C2_3,3,C2_3_1,1,3.0,3,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1133
3,C2,1,C2_3,3,C2_3_1,1,4.0,4,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1133
4,C2,1,C2_3,3,C2_3_1,1,5.0,5,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1133
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
155627,C2,102,C2_3,3,C2_3_102,1,1830.0,1830,1830,1830.0,...,1998.0,1997.0,1998.0,1998.0,0.0,0.0,0.0,0.0,0.0,1834
155628,C2,102,C2_3,3,C2_3_102,1,1831.0,1831,1831,1831.0,...,1996.0,1995.0,1998.0,1995.0,0.0,0.0,0.0,0.0,0.0,1834
155629,C2,102,C2_3,3,C2_3_102,1,1832.0,1832,1832,1832.0,...,1998.0,1998.0,1998.0,1998.0,0.0,0.0,0.0,0.0,0.0,1834
155630,C2,102,C2_3,3,C2_3_102,1,1833.0,1833,1833,1833.0,...,1998.0,1998.0,1998.0,1996.0,0.0,0.0,0.0,0.0,0.0,1834


: 

In [ ]:
df = pd.read_parquet(file_path)
metadata_cols = [x for x in df.columns if "Metadata_" in x]
feature_cols = [x for x in df.columns if "Metadata_" not in x]
features_of_interest = [
    "Nuclei_AreaShape_Area",
    "Nuclei_AreaShape_FormFactor",
    "Nuclei_AreaShape_Eccentricity",
]
df_merged_single_cells = df[metadata_cols + features_of_interest].copy()

# establish outliers in the single-cell profiles by using qc thresholds defined in cosmicqc
cosmicqc.analyze.identify_outliers(
    df=df_merged_single_cells,
    metadata_columns=metadata_cols,
    feature_thresholds={"Nuclei_AreaShape_Area": 1},
)
cosmicqc.analyze.find_outliers(
    df=df_merged_single_cells,
    metadata_columns=metadata_cols,
    feature_thresholds={"Nuclei_AreaShape_FormFactor": 1},
)
cosmicqc.analyze.find_outliers(
    df=df_merged_single_cells,
    metadata_columns=metadata_cols,
    feature_thresholds={"Nuclei_AreaShape_Eccentricity": 1},
)

# label outliers in the single-cell profiles by using qc thresholds defined in cosmicqc
df_labeled_outliers = cosmicqc.analyze.label_outliers(
    df=df_merged_single_cells,
    include_threshold_scores=True,
)

# create a column which indicates whether an erroneous outlier was detected
df_labeled_outliers["analysis.included_at_least_one_outlier"] = df_labeled_outliers[
    [col for col in df_labeled_outliers.columns.tolist() if ".is_outlier" in col]
].any(axis=1)

Number of outliers: 18839 (12.91%)
Outliers Range:
Nuclei_AreaShape_Area Min: 880.0
Nuclei_AreaShape_Area Max: 1814.0
Number of outliers: 3352 (2.30%)
Outliers Range:
Nuclei_AreaShape_FormFactor Min: 0.955119794160055
Nuclei_AreaShape_FormFactor Max: 1.446802115179039
Number of outliers: 20253 (13.88%)
Outliers Range:
Nuclei_AreaShape_Eccentricity Min: 0.7205719261859989
Nuclei_AreaShape_Eccentricity Max: 0.9951065427463287


In [9]:
df_labeled_outliers = df_labeled_outliers["analysis.included_at_least_one_outlier"]
outliers_counts = df_labeled_outliers.value_counts()
outliers_counts

analysis.included_at_least_one_outlier
False    155632
Name: count, dtype: int64

In [11]:
outliers_counts.iloc[0]

np.int64(155632)

In [12]:
# show the percentage of total dataset
print(
    np.round((outliers_counts.iloc[1] / outliers_counts.iloc[0]) * 100, 2),
    "%",
    "of",
    outliers_counts.iloc[0],
    "include erroneous outliers of some kind.",
)

IndexError: single positional indexer is out-of-bounds

In [ ]:
before_shape = df.shape
# df = df.iloc[df_labeled_outliers.index[df_labeled_outliers == False], :]
df = df.loc[~df_labeled_outliers]
print(
    f"Prior to qc we had {before_shape[0]} rows and after removing outliers we have {df.shape[0]} rows."
)

In [ ]:
df.to_parquet(qc_profiles_path, index=False)

In [ ]:
df.head()